# Optuna: spend a tuning budget on useful experiments

A handwritten-digit classifier has several adjustable settings, but training every combination would waste time. You need a defensible objective, a bounded search, a way to stop weak trials, and independent evidence that the selected settings help. Optuna organizes those experiments; it does not choose the right evaluation design for you.

We will tune an incremental linear classifier on scikit-learn’s bundled 8×8 digits, compare against a fixed baseline, inspect pruned trials, resume study metadata from SQLite and save the selected model. All teaching and code are here. **No dataset download, server, GPU or account is required after installation.**

Before every search decision, state what you want to improve and what data is allowed to influence that decision.

## 1. Record the environment and a bounded budget

Use `optuna==4.5.0` with the repository's scikit-learn, numpy, pandas, matplotlib and joblib. These pins fit the Python 3.9 environment. Each run creates a new study name in `outputs/optuna/studies.sqlite3`, preserving earlier studies. Reports and the selected-model file are replaced when rerun.

A **trial** evaluates one configuration. A **study** collects trials for an objective. A **sampler** proposes settings; a **pruner** can stop a trial after intermediate evidence. A sampler seed supports repeatability for this sequential run, not universal determinism for parallel execution.

In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import uuid, json, platform
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import optuna
import sklearn
import joblib
from sklearn.datasets import load_digits
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import SGDClassifier
from sklearn.dummy import DummyClassifier
from sklearn.metrics import log_loss, accuracy_score, f1_score, ConfusionMatrixDisplay
from sklearn.pipeline import Pipeline
from IPython.display import display

OUT = Path('outputs/optuna'); OUT.mkdir(parents=True, exist_ok=True)
SEED = 42; EPOCHS = 20; N_TRIALS = 16
optuna.logging.set_verbosity(optuna.logging.WARNING)
print({'python':platform.python_version(), 'optuna':optuna.__version__, 'sklearn':sklearn.__version__})

## 2. Decide which observations can guide the search

One row is an image represented by 64 pixel values; the target is a digit from 0 through 9. Reserve approximately 20% each for validation and testing, leaving 60% for training. The search repeatedly sees validation performance; it must never use the test labels in its objective.

Fit scaling only on training rows. This demonstration uses a stratified random split; writer identities are not available in this packaged workflow, so it does not establish generalization to unseen writers. For repeated customers or time series, design grouped or chronological splits instead.

In [ ]:
digits = load_digits()
indices = np.arange(len(digits.target))
train_idx, remainder = train_test_split(indices, test_size=0.4, stratify=digits.target, random_state=SEED)
valid_idx, test_idx = train_test_split(remainder, test_size=0.5, stratify=digits.target[remainder], random_state=SEED)
assert not (set(train_idx) & set(valid_idx) or set(train_idx) & set(test_idx) or set(valid_idx) & set(test_idx))
scaler = StandardScaler().fit(digits.data[train_idx])
X_train = scaler.transform(digits.data[train_idx]); y_train = digits.target[train_idx]
X_valid = scaler.transform(digits.data[valid_idx]); y_valid = digits.target[valid_idx]
X_test = scaler.transform(digits.data[test_idx]); y_test = digits.target[test_idx]
classes = np.arange(10)
display(pd.DataFrame({'split':['train','validation','test'], 'rows':[len(train_idx),len(valid_idx),len(test_idx)]}))

## 3. Define “better” before looking for it

We minimize multiclass log loss, which evaluates assigned probabilities and penalizes confident errors. Accuracy evaluates hard class choices and may rank candidates differently. Log loss can be large if a model is confidently wrong; lower is better.

A prior-frequency dummy model provides a probability baseline. Also fix one ordinary SGD configuration before searching, so the selected model is compared with a reasonable untuned learner, not only a trivial one.

In [ ]:
prior = DummyClassifier(strategy='prior').fit(X_train, y_train)
print('Validation prior log loss:', log_loss(y_valid, prior.predict_proba(X_valid), labels=classes))

def make_classifier(params):
    return SGDClassifier(loss='log_loss', learning_rate='constant', eta0=params['eta0'],
                         alpha=params['alpha'], penalty=params['penalty'],
                         l1_ratio=params.get('l1_ratio', 0.15), random_state=SEED)

def fit_epochs(params):
    estimator = make_classifier(params)
    for epoch in range(EPOCHS):
        estimator.partial_fit(X_train, y_train, classes=classes)
    return estimator

fixed_params = {'eta0':0.01, 'alpha':0.0001, 'penalty':'l2'}
fixed_model = fit_epochs(fixed_params)
print('Validation fixed-model log loss:', log_loss(y_valid, fixed_model.predict_proba(X_valid), labels=classes))

## 4. Search ranges should express plausible alternatives

Learning rate and regularization span orders of magnitude, so sample them logarithmically. Penalty is categorical. `l1_ratio` matters only for elastic-net regularization; conditional suggestion keeps it out of L2 trials.

Predict what an excessively large learning rate could do to probability quality. A search cannot discover useful values outside its range, and wider ranges are not automatically better. Keep the search space modest for a 16-trial budget.

In [ ]:
def suggest_parameters(trial):
    params = {
        'eta0': trial.suggest_float('eta0', 1e-4, 0.08, log=True),
        'alpha': trial.suggest_float('alpha', 1e-6, 1e-2, log=True),
        'penalty': trial.suggest_categorical('penalty', ['l2', 'elasticnet'])
    }
    if params['penalty'] == 'elasticnet':
        params['l1_ratio'] = trial.suggest_float('l1_ratio', 0.05, 0.95)
    return params

## 5. Report progress at comparable training steps

Each trial performs up to 20 calls to `partial_fit`. After every epoch it reports validation log loss. `MedianPruner` compares intermediate evidence after startup trials and a warmup period. A pruned trial is intentionally unfinished, not an exception that means the experiment crashed.

The objective returns the final epoch's log loss for completed trials. We do not return the best-ever intermediate score, because refitting that configuration for all 20 epochs would then use a different selection rule. If you select the best epoch instead, save and replay that epoch choice explicitly.

In [ ]:
def objective(trial):
    params = suggest_parameters(trial)
    estimator = make_classifier(params)
    for epoch in range(EPOCHS):
        estimator.partial_fit(X_train, y_train, classes=classes)
        value = log_loss(y_valid, estimator.predict_proba(X_valid), labels=classes)
        if not np.isfinite(value):
            raise ValueError('Non-finite validation loss; inspect the model and data.')
        trial.report(float(value), step=epoch)
        if trial.should_prune():
            raise optuna.TrialPruned(f'Stopped after epoch {epoch+1}')
    trial.set_user_attr('validation_accuracy', float(accuracy_score(y_valid, estimator.predict(X_valid))))
    trial.set_user_attr('nonzero_coefficients', int(np.count_nonzero(estimator.coef_)))
    return float(value)

## 6. Run the study and preserve its history

TPE proposes configurations using previous trials after its startup phase. We enqueue the fixed configuration as a known starting point. SQLite is convenient for this sequential local exercise; large concurrent tuning workloads need a suitable storage backend and coordination.

Before running, predict the maximum number of epochs across all trials: 16 × 20 = 320. Pruning can reduce actual work. Its benefit depends on whether early validation behavior predicts later performance; some slow starters may be discarded.

In [ ]:
storage_url = 'sqlite:///' + str((OUT / 'studies.sqlite3').resolve())
study_name = 'digits-' + uuid.uuid4().hex[:10]
study = optuna.create_study(study_name=study_name, storage=storage_url, direction='minimize',
                           sampler=optuna.samplers.TPESampler(seed=SEED, n_startup_trials=5),
                           pruner=optuna.pruners.MedianPruner(n_startup_trials=4, n_warmup_steps=4))
study.set_user_attr('split_seed', SEED)
study.set_user_attr('dataset', 'sklearn digits; 64 pixel features; stratified 60/20/20')
study.set_user_attr('epochs_per_complete_trial', EPOCHS)
study.enqueue_trial(fixed_params)
study.optimize(objective, n_trials=N_TRIALS, n_jobs=1)
print('Best validation log loss:', study.best_value)
print('Best settings:', study.best_params)
assert len(study.trials) == N_TRIALS

## 7. Distinguish completion, pruning and failure

Inspect trial states before ranking scores. A pruned trial's stored value may describe only its last intermediate step; do not compare it as a completed 20-epoch result. Conditional parameters appear missing for trials where they were irrelevant, not because logging failed.

Count reported epochs and compare with the maximum budget. This counts completed training steps, not wall-clock savings: database and evaluation overhead also matter. A particular seeded run need not produce every possible trial state.

In [ ]:
trials = study.trials_dataframe()
display(trials[['number','value','state','params_alpha','params_eta0','params_penalty']])
print(trials.state.value_counts())
reported_steps = sum(len(t.intermediate_values) for t in study.trials)
print({'reported_epochs':reported_steps, 'maximum_epochs':N_TRIALS*EPOCHS})
complete = [t for t in study.trials if t.state == optuna.trial.TrialState.COMPLETE]
assert complete and np.isclose(study.best_value, min(t.value for t in complete))
fig, axes = plt.subplots(1,2,figsize=(11,4))
finished = trials[trials.state == 'COMPLETE'].sort_values('number')
axes[0].scatter(finished.number, finished.value)
axes[0].plot(finished.number, finished.value.cummin(), label='Best completed value')
axes[0].set(xlabel='Trial', ylabel='Validation log loss'); axes[0].legend()
for trial in study.trials:
    axes[1].plot(list(trial.intermediate_values), list(trial.intermediate_values.values()), alpha=0.5)
axes[1].set(xlabel='Epoch index', ylabel='Validation log loss', title='Intermediate curves')
plt.tight_layout(); plt.savefig(OUT / 'search_history.png'); plt.show()

## 8. Reopen the study without silently rerunning experiments

`load_study` retrieves recorded trials. It does not retrain the winning model, restore arbitrary Python state, or automatically preserve the sampler's random-number state for an identical continuation. A resumed adaptive search can be valid without duplicating the exact uninterrupted proposal sequence.

We reopen and check its identity and best result. Calling `optimize` again would add trials to this study; it would not replace the existing ones. Record a new study when the data split or objective meaning changes.

In [ ]:
reopened = optuna.load_study(study_name=study_name, storage=storage_url)
assert len(reopened.trials) == len(study.trials)
assert reopened.best_params == study.best_params
assert reopened.best_value == study.best_value
print('Preserved study:', reopened.study_name)

## 9. Recreate the selected training protocol before touching test data

Optuna preserves parameters and values, not a fitted estimator by default. Refit the selected settings using the same scaled training rows, seed, epoch count and update order. Compare the reproduced validation score against the recorded objective.

We keep the training population unchanged rather than silently refitting on training plus validation with a different scaling and optimization trajectory. In production, a final refit policy can be appropriate, but define and validate that policy explicitly.

In [ ]:
selected = fit_epochs(study.best_params)
reproduced = log_loss(y_valid, selected.predict_proba(X_valid), labels=classes)
assert np.isclose(reproduced, study.best_value, rtol=1e-8, atol=1e-8)
rows = []
for name, estimator in [('class prior',prior), ('fixed SGD',fixed_model), ('Optuna-selected SGD',selected)]:
    probabilities = estimator.predict_proba(X_test)
    predicted = estimator.predict(X_test)
    rows.append({'model':name, 'test_log_loss':log_loss(y_test, probabilities, labels=classes),
                 'test_accuracy':accuracy_score(y_test,predicted), 'test_macro_f1':f1_score(y_test,predicted,average='macro')})
metrics = pd.DataFrame(rows)
display(metrics)
ConfusionMatrixDisplay.from_predictions(y_test, selected.predict(X_test))
plt.tight_layout(); plt.show()

## 10. Interpret a tuned score without claiming a guaranteed improvement

The chosen trial minimizes observed validation loss among the tried configurations. It is not a proof of a global optimum, and searching more trials can overfit the same validation set. If test performance is worse than the fixed baseline, report that rather than redefining the objective afterward.

For a larger project, use cross-validation inside each objective or repeated development splits, with preprocessing fitted independently inside each fold. Nested cross-validation separates tuning from performance estimation. Reusing one scaler fitted on all rows would leak held-out information into those folds.

Optuna's parameter importance estimates can help describe a completed study, but a small conditional search gives unstable importance and is not a causal analysis. Inspect ranges, completed counts and interactions before narrowing the next study.

In the checked run, the selected model had test log loss about 0.2851, versus 0.2623 for the fixed SGD model; lower is better. This is useful negative evidence: minimizing this validation objective did not guarantee better held-out probabilities. Keep that result rather than tuning until the test comparison looks favorable. Values may vary with environment versions.

## 11. See why two goals produce a set of tradeoffs

Suppose one design variable should be near 0 for one goal and near 2 for another. There is no single value minimizing both squared distances. This small mathematical example demonstrates a Pareto front separately from the classifier; it is not a measured latency benchmark.

With multiple directions, inspect `best_trials` rather than a single `best_trial`. The caller must choose a tradeoff or impose a constraint. Intermediate reporting/pruning used in the earlier single-objective study is not applied here.

In [ ]:
def two_goals(trial):
    x = trial.suggest_float('x', -1, 3)
    return x*x, (x-2)*(x-2)
tradeoff = optuna.create_study(directions=['minimize','minimize'], sampler=optuna.samplers.NSGAIISampler(seed=SEED))
tradeoff.optimize(two_goals, n_trials=16)
front = pd.DataFrame([{'x':t.params['x'], 'goal_1':t.values[0], 'goal_2':t.values[1]} for t in tradeoff.best_trials])
display(front.sort_values('x'))
assert len(front) > 0

## 12. Hand off the estimator together with the preprocessing

Store a pipeline containing the fitted scaler and selected classifier. Reload and compare probabilities on fixed raw digit rows. Only load joblib artifacts from trusted sources; they use Python serialization. Save split IDs and the study name so that a later reader can distinguish the data, search and trained artifact.

A notebook rerun creates a new study, while the report files below are replaced. The metadata records which study produced this particular saved pipeline.

In [ ]:
selected_pipeline = Pipeline([('scale',scaler), ('classifier',selected)])
joblib.dump(selected_pipeline, OUT / 'selected_pipeline.joblib')
reloaded = joblib.load(OUT / 'selected_pipeline.joblib')
assert np.allclose(reloaded.predict_proba(digits.data[test_idx[:8]]), selected.predict_proba(X_test[:8]))
trials.to_csv(OUT / 'trials.csv', index=False)
metrics.to_csv(OUT / 'test_metrics.csv', index=False)
np.savez(OUT / 'split_indices.npz', train=train_idx, validation=valid_idx, test=test_idx)
(OUT / 'study_metadata.json').write_text(json.dumps({'study_name':study_name, 'storage':storage_url,
    'best_params':study.best_params, 'validation_log_loss':study.best_value, 'seed':SEED,
    'epochs':EPOCHS, 'optuna':optuna.__version__}, indent=2))
print('Saved artifacts:', sorted(p.name for p in OUT.iterdir()))

## 13. Explain what the evidence supports

1. Before a new search, halve the trial budget and predict which conclusions become less stable. Compare development results across several prespecified seeds without repeatedly redesigning against this test set.
2. Disable pruning in a separate study and compare training steps and validation results. A saved epoch is not automatically a saved second.
3. Change the objective from log loss to macro F1 in a new study. Explain why direction becomes `maximize` and why model rankings may change.
4. Design a grouped or temporal split for your own application before adding more hyperparameters.

| Observation | Investigate |
|---|---|
| Many failed trials | Actual exceptions, invalid configurations and non-finite metrics |
| No pruned trials | Warmup, number of completed trials and learning-curve similarity |
| Winning trial cannot be reproduced | Seed, scaler, update order, epochs and data snapshot |
| SQLite locking during parallel work | Storage suitability and worker coordination |
| Excellent validation but weak test | Validation overfitting, split variance and distribution shift |

Official references: [study creation](https://optuna.readthedocs.io/en/v4.5.0/reference/generated/optuna.create_study.html), [samplers and pruning](https://optuna.readthedocs.io/en/v4.5.0/tutorial/10_key_features/003_efficient_optimization_algorithms.html), [persistent studies](https://optuna.readthedocs.io/en/v4.5.0/tutorial/20_recipes/001_rdb.html), and [multi-objective optimization](https://optuna.readthedocs.io/en/v4.5.0/tutorial/20_recipes/002_multi_objective.html).

## Second project: spend an equal search budget on temporal validation

A subscription-scoring export has repeated rows, malformed and missing usage, missing labels, a new category appearing late, and a tempting field populated only after the outcome. Build a time-ordered development/test workflow on 1,000 generated records with a mild late-period shift. This is a synthetic operational scenario, not a real churn dataset; it is designed to expose boundary mistakes while keeping training small.

Run these cells in order. This extension uses its own `p2_` variables and writes only its named outputs under `outputs/project2/`. Predict each comparison before running it; the worked discussion after the code explains the result and its limits.

### Work through a feature that predicts too well

`refund_after_outcome` is generated from the label. It would look highly useful but is unavailable when a churn prediction is made. Exclude it by an explicit feature allowlist before preprocessing or tuning. Also exclude row IDs, which index time but are not intended predictors here. Missing labels cannot be imputed as though they were ordinary features.

Fit imputation, scaling and categorical vocabulary only on the training period. An unseen late category should be handled deliberately rather than silently refitting the encoder on the test set.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import time
from IPython.display import display
p2_out=Path('outputs/project2');p2_out.mkdir(parents=True,exist_ok=True)

p2_rng=np.random.default_rng(63);p2_n=1000
p2_usage=p2_rng.gamma(2,3,p2_n);p2_tenure=p2_rng.integers(1,36,p2_n)
p2_region=p2_rng.choice(['East','West','North'],p2_n).astype(object);p2_region[900:]='NewRegion'
p2_logit=1.8-.28*p2_usage-.035*p2_tenure+.5*(p2_region=='West')+.3*(np.arange(p2_n)>=800)
p2_target=p2_rng.binomial(1,1/(1+np.exp(-p2_logit)))
p2_raw=pd.DataFrame({'row_id':np.arange(p2_n),'usage':p2_usage.astype(object),'tenure':p2_tenure,
 'region':p2_region,'churn':p2_target.astype(float),'refund_after_outcome':p2_target+np.random.default_rng(64).normal(0,.01,p2_n)})
p2_raw.loc[10:19,'usage']='malformed';p2_raw.loc[20:39,'usage']=None;p2_raw.loc[3:7,'churn']=np.nan
p2_raw=pd.concat([p2_raw,p2_raw.iloc[-20:]],ignore_index=True)
p2_clean=p2_raw.drop_duplicates().copy();assert p2_clean.row_id.is_unique
p2_clean['usage']=pd.to_numeric(p2_clean.usage,errors='coerce')
p2_missing_label=int(p2_clean.churn.isna().sum());p2_clean=p2_clean.dropna(subset=['churn'])
p2_clean['churn']=p2_clean.churn.astype(int)
p2_train=p2_clean[p2_clean.row_id<600];p2_valid=p2_clean[(p2_clean.row_id>=600)&(p2_clean.row_id<800)];p2_test=p2_clean[p2_clean.row_id>=800]
p2_features=['usage','tenure','region'];assert 'refund_after_outcome' not in p2_features
assert len(p2_train)==595 and len(p2_valid)==len(p2_test)==200
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler,OneHotEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.dummy import DummyClassifier
from sklearn.metrics import log_loss,roc_auc_score,average_precision_score

def p2_preprocess():
    return ColumnTransformer([('numeric',Pipeline([('impute',SimpleImputer(strategy='median',add_indicator=True)),('scale',StandardScaler())]),['usage','tenure']),
      ('category',OneHotEncoder(handle_unknown='ignore',sparse_output=False),['region'])],sparse_threshold=0)
def p2_pipeline(estimator):return Pipeline([('prepare',p2_preprocess()),('model',estimator)])

import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)
from sklearn.model_selection import TimeSeriesSplit
p2_development=p2_clean[p2_clean.row_id<800]
p2_folds=list(TimeSeriesSplit(n_splits=3).split(p2_development))
assert all(a.max()<b.min() for a,b in p2_folds)
def p2_objective(trial):
    c_value=trial.suggest_float('C',1e-3,100,log=True)
    losses=[]
    for fold,(a,b) in enumerate(p2_folds):
        pipe=p2_pipeline(LogisticRegression(C=c_value,max_iter=1000,random_state=6))
        pipe.fit(p2_development.iloc[a][p2_features],p2_development.iloc[a].churn)
        losses.append(log_loss(p2_development.iloc[b].churn,pipe.predict_proba(p2_development.iloc[b][p2_features])[:,1]))
    trial.set_user_attr('fold_losses',losses)
    return float(np.mean(losses))
p2_studies={};p2_budget=[]
for p2_name,p2_sampler in [('random',optuna.samplers.RandomSampler(seed=8)),('TPE',optuna.samplers.TPESampler(seed=8,n_startup_trials=3))]:
    p2_study=optuna.create_study(direction='minimize',sampler=p2_sampler,pruner=optuna.pruners.NopPruner())
    p2_t=time.perf_counter();p2_study.optimize(p2_objective,n_trials=8,n_jobs=1)
    p2_budget.append({'sampler':p2_name,'best_cv_loss':p2_study.best_value,'model_fits':8*3,'seconds':time.perf_counter()-p2_t,'C':p2_study.best_params['C']})
    p2_studies[p2_name]=p2_study
p2_budget=pd.DataFrame(p2_budget).sort_values(['best_cv_loss','sampler']);display(p2_budget)

### Work through the hidden cost of one trial

Eight trials with three expanding-window folds mean 24 model fits per sampler, not eight. Every fold fits its own preprocessing on earlier rows. We deliberately disable pruning here: a partial average over easy early folds is not necessarily comparable with the complete temporal objective.

Choose the best development result across the two fixed eight-trial searches, refit on the full development period, then inspect the final test once. Comparing samplers across repeated final tests would turn the test into another tuning surface.

In [ ]:
p2_winner=p2_budget.iloc[0]
p2_best=p2_pipeline(LogisticRegression(C=float(p2_winner.C),max_iter=1000,random_state=6))
p2_best.fit(p2_development[p2_features],p2_development.churn)
p2_prob=p2_best.predict_proba(p2_test[p2_features])[:,1]
p2_results={'selected_sampler':p2_winner.sampler,'development_cv_loss':float(p2_winner.best_cv_loss),
 'test_log_loss':float(log_loss(p2_test.churn,p2_prob)),'test_auc':float(roc_auc_score(p2_test.churn,p2_prob)),'total_search_fits':48}
assert sum(len(s.trials) for s in p2_studies.values())==16
print(p2_results)
p2_budget.to_csv(p2_out/'optuna_equal_budget.csv',index=False)
import json
(p2_out/'optuna_temporal_result.json').write_text(json.dumps(p2_results,indent=2))

### Explain why this is not a sampler league table

One seed, one search space and eight trials are insufficient to declare TPE or random search universally better. The comparison demonstrates equal budgets and a leakage-aware objective. Time and score differences may be small; describe the actual run rather than assuming an adaptive sampler must win.

**Try:** increase `n_startup_trials` beyond eight. **Worked reasoning:** TPE has no adaptive phase within this budget, so interpreting the result as a strong test of TPE's learned proposals would be misleading. If training becomes expensive, choose pruning evidence whose step meaning is comparable across trials, and report completed versus pruned work.